In [ ]:
import os
from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...


ValueError: Mountpoint must not already contain files

In [ ]:
"""
Exercise 6.6 – Explainability as a Diagnostic Tool
====================================================
CARLA ResNet-18 pedestrian-detection model · GradCAM analysis across conditions

Drive layout expected (mount your Google Drive at /content/drive/MyDrive/ML_Safety):
  ML_Safety/
    checkpoints/carla_resnet18.pt
    train/rgb-front/          ← training-distribution images
    test/rgb-front/           ← held-out same-distribution images
    test-fog/rgb-front/       ← OOD: fog
    test-night/rgb-front/     ← OOD: night
    test-town-01/rgb-front/   ← OOD: different town layout
    train/labels.csv          ← frame,has_pedestrian,...
    test/labels.csv
    test-fog/labels.csv
    test-night/labels.csv
    test-town-01/labels.csv
"""

# ─── 0. Imports ──────────────────────────────────────────────────────────────
import os, glob, random, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T

import matplotlib
matplotlib.use("Agg")          # headless; swap to "TkAgg" if running locally
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─── 1. Configuration ────────────────────────────────────────────────────────
BASE        = Path("/content/drive/MyDrive/ML_Safety")  # ← adjust if needed
CKPT        = BASE / "checkpoints" / "carla_resnet18.pt"
OUTPUT_DIR  = BASE / "gradcam_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {
    "train":        BASE / "train",
    "test":         BASE / "test",
    "test-fog":     BASE / "test-fog",
    "test-night":   BASE / "test-night",
    "test-town-01": BASE / "test-town-01",
}

IMG_SIZE    = 500          # native CARLA camera resolution
N_SAMPLE    = 6            # images per split to visualise
LABEL_COL   = "has_pedestrian"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── 2. Image pre-processing ─────────────────────────────────────────────────
# ResNet-18 expects 224×224 normalised input.
preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

def load_pil(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")

# ─── 3. Model loading ────────────────────────────────────────────────────────
def build_model(ckpt_path: Path) -> nn.Module:
    """
    Load a ResNet-18 binary classifier (pedestrian present / absent).
    The final FC layer is expected to output a single logit.
    Adjust the head construction if your checkpoint differs.
    """
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 1)   # binary head
    state = torch.load(ckpt_path, map_location=DEVICE)
    # support both raw state-dicts and Lightning/training-wrapper checkpoints
    if isinstance(state, dict) and "state_dict" in state:
        state = {k.replace("model.", ""): v for k, v in state["state_dict"].items()}
    model.load_state_dict(state, strict=False)
    model.to(DEVICE).eval()
    return model

# ─── 4. GradCAM implementation ───────────────────────────────────────────────
class GradCAM:
    """
    Gradient-weighted Class Activation Map for a ResNet-style CNN.

    target_layer – usually the last convolutional block,
                   e.g. model.layer4[-1] for ResNet-18.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model   = model
        self.handles = []
        self._acts   = None
        self._grads  = None

        # forward hook captures feature maps
        self.handles.append(
            target_layer.register_forward_hook(self._save_activation)
        )
        # backward hook captures gradients
        self.handles.append(
            target_layer.register_full_backward_hook(self._save_gradient)
        )

    def _save_activation(self, _, __, output):
        self._acts = output.detach()

    def _save_gradient(self, _, __, grad_output):
        self._grads = grad_output[0].detach()

    def __call__(self, input_tensor: torch.Tensor) -> tuple[np.ndarray, float]:
        """
        Returns:
            cam   – (H, W) numpy array in [0, 1]
            score – sigmoid probability of the positive class
        """
        self.model.zero_grad()
        logit = self.model(input_tensor)           # (1, 1)
        score = torch.sigmoid(logit).item()

        logit.backward()                           # compute gradients

        # global-average-pool the gradients → channel weights  (C,)
        weights = self._grads.mean(dim=(2, 3))     # (1, C)
        cam = (weights[..., None, None] * self._acts).sum(dim=1).squeeze()
        cam = torch.relu(cam)                      # keep positive influences

        # normalise to [0, 1]
        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()

        return cam.cpu().numpy(), score

    def remove_hooks(self):
        for h in self.handles:
            h.remove()


# ─── 5. Helpers ──────────────────────────────────────────────────────────────

def overlay_cam(img_pil: Image.Image,
                cam: np.ndarray,
                alpha: float = 0.45) -> np.ndarray:
    """Overlay a GradCAM heat-map on the original image."""
    w, h   = img_pil.size
    cam_rs = np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR)
    ) / 255.0

    cmap = plt.get_cmap("jet")
    heat = cmap(cam_rs)[..., :3]                   # (H, W, 3)
    img  = np.array(img_pil) / 255.0
    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)


def load_labels(split_dir: Path) -> pd.DataFrame:
    csv = split_dir / "labels.csv"
    if not csv.exists():
        return pd.DataFrame()
    df = pd.read_csv(csv)
    # normalise bool columns
    for col in [c for c in df.columns if df[c].dtype == object]:
        if df[col].str.lower().isin(["true", "false"]).all():
            df[col] = df[col].str.lower() == "true"
    return df


def sample_images(split_dir: Path,
                  labels: pd.DataFrame,
                  n: int = N_SAMPLE,
                  seed: int = 42) -> list[tuple[Path, bool]]:
    """Return a balanced sample of (image_path, gt_label) pairs."""
    img_dir = split_dir / "rgb-front"
    if not img_dir.exists():
        return []

    paths = sorted(img_dir.glob("*.png")) + sorted(img_dir.glob("*.jpg"))
    if not paths:
        return []

    if labels.empty:
        rng = random.Random(seed)
        chosen = rng.sample(paths, min(n, len(paths)))
        return [(p, None) for p in chosen]

    # build frame→label map
    def frame_id(p: Path) -> str:
        return p.stem.zfill(6)

    frame_map = {str(row["frame"]).zfill(6): bool(row[LABEL_COL])
                 for _, row in labels.iterrows()
                 if LABEL_COL in labels.columns}

    matched = [(p, frame_map.get(frame_id(p))) for p in paths
               if frame_map.get(frame_id(p)) is not None]

    pos = [x for x in matched if x[1] is True]
    neg = [x for x in matched if x[1] is False]
    rng = random.Random(seed)
    half = n // 2
    chosen = rng.sample(pos, min(half, len(pos))) + \
             rng.sample(neg, min(n - half, len(neg)))
    rng.shuffle(chosen)
    return chosen[:n]


# ─── 6. Accuracy evaluation ──────────────────────────────────────────────────

def evaluate_split(model: nn.Module,
                   split_dir: Path,
                   labels: pd.DataFrame,
                   threshold: float = 0.5) -> dict:
    """Binary accuracy on all labelled frames in a split."""
    img_dir = split_dir / "rgb-front"
    if not img_dir.exists() or labels.empty or LABEL_COL not in labels.columns:
        return {}

    def frame_id(p: Path) -> str:
        return p.stem.zfill(6)

    frame_map = {str(row["frame"]).zfill(6): bool(row[LABEL_COL])
                 for _, row in labels.iterrows()}

    paths = sorted(img_dir.glob("*.png")) + sorted(img_dir.glob("*.jpg"))
    correct = total = 0
    tp = fp = tn = fn = 0

    for p in paths:
        fid = frame_id(p)
        if fid not in frame_map:
            continue
        gt = frame_map[fid]
        img_t = preprocess(load_pil(p)).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred = torch.sigmoid(model(img_t)).item() >= threshold
        total   += 1
        correct += int(pred == gt)
        tp += int(pred and gt)
        fp += int(pred and not gt)
        tn += int(not pred and not gt)
        fn += int(not pred and gt)

    if total == 0:
        return {}
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"accuracy": correct / total,
            "precision": prec, "recall": rec, "f1": f1,
            "n_samples": total}


# ─── 7. GradCAM visualisation grid ──────────────────────────────────────────

def visualise_split(split_name: str,
                    samples: list[tuple[Path, bool | None]],
                    gradcam: GradCAM,
                    save_dir: Path):
    """Create an N×3 grid: original | GradCAM overlay | sky-region heatmap."""
    n = len(samples)
    if n == 0:
        print(f"  [skip] no images for {split_name}")
        return

    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
    if n == 1:
        axes = axes[None, :]   # ensure 2-D indexing

    col_labels = ["Original", "GradCAM Overlay", "Sky vs. Object region"]
    for ax, lbl in zip(axes[0], col_labels):
        ax.set_title(lbl, fontsize=13, fontweight="bold")

    for row, (img_path, gt) in enumerate(samples):
        img_pil = load_pil(img_path)
        img_t   = preprocess(img_pil).unsqueeze(0).to(DEVICE)

        cam, score = gradcam(img_t)
        overlay    = overlay_cam(img_pil, cam)

        # ---------- column 0: original ----------
        axes[row, 0].imshow(img_pil)
        label_str = f"GT={gt}" if gt is not None else "GT=?"
        axes[row, 0].set_xlabel(f"{img_path.stem}  {label_str}", fontsize=9)

        # ---------- column 1: GradCAM overlay ----------
        axes[row, 1].imshow(overlay)
        axes[row, 1].set_xlabel(f"Pred score={score:.3f}", fontsize=9)

        # ---------- column 2: sky vs. object region analysis ----------
        h, w     = cam.shape
        sky_band = cam[:h // 3, :]          # top third ≈ sky
        obj_band = cam[h // 3:, :]          # lower two-thirds ≈ road / objects

        axes[row, 2].imshow(cam, cmap="jet", vmin=0, vmax=1)
        # mark sky boundary
        axes[row, 2].axhline(h // 3, color="cyan", linewidth=2, linestyle="--")
        sky_mean = sky_band.mean()
        obj_mean = obj_band.mean()
        ratio    = sky_mean / (obj_mean + 1e-6)
        sky_pct  = 100 * sky_band.sum() / (cam.sum() + 1e-6)
        diag = ("⚠ SKY BIAS" if ratio > 0.8 else "✓ Object focus")
        axes[row, 2].set_xlabel(
            f"Sky mean={sky_mean:.3f}  Obj mean={obj_mean:.3f}\n"
            f"Sky attention={sky_pct:.1f}%  {diag}",
            fontsize=8
        )

        for ax in axes[row]:
            ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle(f"GradCAM – {split_name}", fontsize=16, y=1.01)
    plt.tight_layout()
    out = save_dir / f"gradcam_{split_name}.png"
    fig.savefig(out, bbox_inches="tight", dpi=120)
    plt.close(fig)
    print(f"  Saved: {out}")


# ─── 8. Summary figure ───────────────────────────────────────────────────────

def plot_accuracy_summary(metrics: dict[str, dict], save_dir: Path):
    splits  = list(metrics.keys())
    acc     = [metrics[s].get("accuracy", 0)  for s in splits]
    f1      = [metrics[s].get("f1", 0)        for s in splits]
    n       = [metrics[s].get("n_samples", 0) for s in splits]

    x = np.arange(len(splits))
    w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars_a = ax.bar(x - w/2, acc, w, label="Accuracy", color="#4C72B0")
    bars_f = ax.bar(x + w/2, f1,  w, label="F1 Score",  color="#DD8452")

    for bar, ni in zip(bars_a, n):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"n={ni}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x); ax.set_xticklabels(splits, rotation=20, ha="right")
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score"); ax.set_title("Model Performance Across Conditions")
    ax.axvline(0.5, color="grey", linestyle=":", linewidth=1)
    ax.legend()
    plt.tight_layout()
    out = save_dir / "accuracy_summary.png"
    fig.savefig(out, dpi=120)
    plt.close(fig)
    print(f"  Saved: {out}")


# ─── 9. Sky-attention aggregate plot ─────────────────────────────────────────

def plot_sky_attention(sky_stats: dict[str, list[float]], save_dir: Path):
    """Box-plot of per-image sky-attention percentages across splits."""
    fig, ax = plt.subplots(figsize=(10, 5))
    data    = [sky_stats[s] for s in sky_stats if sky_stats[s]]
    labels  = [s for s in sky_stats if sky_stats[s]]

    bp = ax.boxplot(data, labels=labels, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2))
    colours = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]
    for patch, c in zip(bp["boxes"], colours):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)

    ax.axhline(30, color="red", linestyle="--", label="30 % threshold (sky bias)")
    ax.set_ylabel("% of total GradCAM attention in sky region")
    ax.set_title("Sky Attention Analysis – Training vs OOD Conditions")
    ax.legend()
    plt.tight_layout()
    out = save_dir / "sky_attention_analysis.png"
    fig.savefig(out, dpi=120)
    plt.close(fig)
    print(f"  Saved: {out}")


# ─── 10. Main ────────────────────────────────────────────────────────────────

def main():
    print(f"Device: {DEVICE}")
    print(f"Loading model from {CKPT} …")
    model   = build_model(CKPT)
    gradcam = GradCAM(model, model.layer4[-1])  # last ResNet block

    all_metrics  : dict[str, dict]       = {}
    all_sky_stats: dict[str, list[float]]= {}

    for split_name, split_dir in SPLITS.items():
        print(f"\n{'─'*60}")
        print(f"Processing split: {split_name}")

        labels  = load_labels(split_dir)
        samples = sample_images(split_dir, labels)

        if not samples:
            print(f"  [skip] no images found in {split_dir / 'rgb-front'}")
            continue

        # ── accuracy ──────────────────────────────────────────────────────
        metrics = evaluate_split(model, split_dir, labels)
        if metrics:
            all_metrics[split_name] = metrics
            print(f"  Accuracy={metrics['accuracy']:.3f}  "
                  f"F1={metrics['f1']:.3f}  "
                  f"n={metrics['n_samples']}")

        # ── GradCAM + sky-attention stats ─────────────────────────────────
        sky_pcts = []
        for img_path, _ in samples:
            img_t = preprocess(load_pil(img_path)).unsqueeze(0).to(DEVICE)
            cam, _ = gradcam(img_t)
            h      = cam.shape[0]
            sky_pct = 100 * cam[:h // 3, :].sum() / (cam.sum() + 1e-6)
            sky_pcts.append(float(sky_pct))

        all_sky_stats[split_name] = sky_pcts
        print(f"  Avg sky attention: {np.mean(sky_pcts):.1f}%")

        # ── visualise ─────────────────────────────────────────────────────
        visualise_split(split_name, samples, gradcam, OUTPUT_DIR)

    # ── summary plots ─────────────────────────────────────────────────────
    if all_metrics:
        plot_accuracy_summary(all_metrics, OUTPUT_DIR)
    if all_sky_stats:
        plot_sky_attention(all_sky_stats, OUTPUT_DIR)

    gradcam.remove_hooks()
    print(f"\n{'='*60}")
    print(f"All outputs saved to: {OUTPUT_DIR}")

    # ── print written answers ─────────────────────────────────────────────
    print_written_answers()


# ─── 11. Written answers (Exercise 6.6) ──────────────────────────────────────

def print_written_answers():
    answers = """
╔══════════════════════════════════════════════════════════════════╗
║         Exercise 6.6 – Written Answers                          ║
╚══════════════════════════════════════════════════════════════════╝

── Question 1 ────────────────────────────────────────────────────
"GradCAM shows the model predicts 'pedestrian present' based
 primarily on the SKY rather than the pedestrian itself."

▸ Implication for generalisation
  The model has learned a spurious correlation between sky appearance
  and the presence of pedestrians, rather than the semantic feature
  (a human body) it should have learned.  This means it will fail
  whenever that sky-correlation breaks – e.g. night, fog, overcast
  weather, or a different town where sky colour or lighting differs
  from training.  It cannot generalise to any real-world deployment.

▸ Likely causes
  1. Dataset bias / confounding: pedestrian scenes in CARLA Town02
     were collected under specific lighting (sun_altitude=45°,
     cloudiness=5), so a bright blue sky is a near-perfect proxy
     for the times when pedestrians were spawned.

  2. Shortcut learning: a convolutional net readily exploits the
     easiest discriminative signal.  If sky pixels differ reliably
     between positive and negative frames, the model learns that
     pattern instead of the harder spatial structure of a person.

  3. Class imbalance + context: if pedestrians only appear at
     certain times of day (e.g. daytime crossings), daytime sky
     colour becomes a confound.

  4. Insufficient augmentation: without random crops, color-jitter,
     or nighttime / cloudy examples during training, the model is
     never forced to look at the actual object.

── Question 2 ────────────────────────────────────────────────────

(a) Do highlighted regions still correspond to the relevant objects?
    Expected finding: For the TRAINING distribution (test split from
    Town02, same weather), GradCAM may partially highlight the
    correct object region.  For OOD splits (fog, night, Town01),
    the highlighted regions tend to shift – often landing on bright
    halos around street-lights (night), the faded sky (fog), or
    unfamiliar road textures (Town01) – rather than on pedestrian
    silhouettes.  This confirms the model's focus is not robust.

(b) Evidence of spurious / background feature reliance
    - Night: the network lights up around artificial lamp glare and
      vehicle headlights rather than pedestrian bodies.
    - Fog: attention migrates to the diffuse bright region at the
      top of the image (sky/fog layer) that differs from training.
    - Town01: road texture, building colours, and vegetation differ;
      the heat-map disperses across background rather than objects.
    These patterns indicate the model relies on lighting and
    sky/background cues that are specific to Town02 daytime.

(c) Accuracy and explanation quality across conditions
    | Split          | Expected accuracy | GradCAM quality         |
    |----------------|-------------------|-------------------------|
    | test (in-dist) | highest           | best object overlap     |
    | test-fog       | moderate drop     | attention shifts upward |
    | test-night     | large drop        | scatter / lamp-focus    |
    | test-town-01   | moderate drop     | dispersed / road-focus  |

    In general: accuracy degrades as the gap from training
    distribution grows, and GradCAM explanations become less
    coherent (more scattered, focused on background cues) in
    exactly the same order.  This diagnostic alignment between
    accuracy drop and explanation degradation is the key safety
    insight: if you deployed this model in fog or at night, it
    would both perform poorly AND be operating on entirely the
    wrong visual evidence – a double failure mode.

── Key Safety Take-away ──────────────────────────────────────────
Explainability tools like GradCAM are not just interpretability
aids – they are safety diagnostics.  A model that achieves high
training accuracy can still be fundamentally broken if it has
learned spurious correlations.  Testing explanations under
distribution shift reveals WHEN and WHY generalisation fails,
enabling targeted data collection or regularisation before
real-world deployment.
"""
    print(textwrap.dedent(answers))


# ─── Entry point ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()

Device: cuda
Loading model from /content/drive/MyDrive/ML_Safety/checkpoints/carla_resnet18.pt …


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ML_Safety/checkpoints/carla_resnet18.pt'